# Step 2 — Filter the raw videos

Pipeline stage 2. Stage 1 (`get_raw_videos.ipynb`) searched YouTube for `"[CEO] interview"`
once per **CEO/year** pair and kept *everything* the API returned — including a lot of videos
that have nothing to do with the CEO (a search for `"Erik Carlson interview"` happily returns
Tucker Carlson).

**Both layers of this notebook look at the TITLE and nothing else.** The description is not
searched at either stage. A description is uploader-controlled boilerplate as often as it is
content — affiliate disclosures, show branding, copyright notices, a paste of the channel's
standard blurb — and matching on it is what let the two long tails of off-topic material into
the earlier version of this set.

## Layer 1 — is this video about *this CEO*?

A title is kept when **either** of these holds:

| path | the title contains | example |
|---|---|---|
| **CEO name** | any spelling of the CEO's name | *"**Doug McMillon** on tenure"* |
| **company + CEO marker** | a company spelling **and** the word `CEO` or `chief executive officer`, anywhere in the title and in any order | *"**Walmart CEO** steps down"* |

The marker requirement is the change that matters. A bare company name in a title says nothing
about whether the CEO is in the video: *"How Walmart is trying to be something for everyone"*,
*"Chevron Investor Day 2025 Recap"* and *"Amazon Just Revealed Their AI Master Plan"* all
matched under the old company-only rule and none of them is a CEO video. Requiring the marker
drops **3,818** such titles. The two markers are not adjacency-checked — `CEO` may sit anywhere
relative to the company name — and no possessive handling is needed, because normalisation
turns `Walmart's` into `walmart` before the comparison.

Name spellings come from these columns:

| source column | notes |
|---|---|
| `CEO` / `second_CEO` | the canonical name |
| `CEO_alt_names` / `second_CEO_alt_names` | `;`-separated list of spellings |
| `company` | |
| `company_alt_names` | the legal/long form, e.g. `IBM` → `International Business Machines` |

`second_CEO` is a co-CEO: stage 1 ran a **separate search** for them, stored as its own record
with `ceo_slot = "second"`. Those records are filtered against the `second_*` columns and are
treated throughout as ordinary CEOs — "CEO or second_CEO" is always one population.

**Input:** `data/output/videos_metadata/all_videos.jsonl`
**Output:** `data/output/videos_metadata/filtered_videos.jsonl` — same record shape, with
`videos` reduced to the ones that matched (plus a `match` block on each kept video saying
*what* matched, so any decision here can be audited later).

## 1. Configuration

In [1]:
import json
import re
import unicodedata
import collections
from pathlib import Path
from statistics import mean, median

import pandas as pd
import isodate

# ── Paths ────────────────────────────────────────────────────────────────────
CEOS_CSV    = Path("data/ceos.csv")
OUTPUT_DIR  = Path("data/output/videos_metadata")
INPUT_JSONL = OUTPUT_DIR / "all_videos.jsonl"
OUTPUT_JSONL = OUTPUT_DIR / "filtered_videos.jsonl"

# ── Filter behaviour ─────────────────────────────────────────────────────────
# A title is kept if it names the CEO, OR names the company AND carries a CEO marker.
# A bare company name is not enough: "Chevron Investor Day 2025 Recap" is not a CEO video.
MATCH_ON_CEO     = True   # CEO / second_CEO + their alt names — sufficient on its own
MATCH_ON_COMPANY = True   # company + company_alt_names — only together with a marker below

# The marker may sit anywhere in the title, in any order relative to the company name.
CEO_MARKERS = ["ceo", "chief executive officer"]

ALT_SEPARATOR = ";"       # CEO_alt_names / second_CEO_alt_names are ";"-separated lists

## 2. Normalising titles and names

`"Walmart CEO Doug McMillon on tenure"` has to match `Doug McMillon`, and
`"Björn Gulden — adidas"` has to match `Bjorn Gulden`. So both sides go through the same
normalisation before comparison:

1. strip accents (`é` → `e`) — YouTube titles are inconsistent about them;
2. lowercase;
3. `&` → `and`, so `AT&T` matches `AT and T` and `Procter & Gamble` matches either spelling;
4. every other non-alphanumeric character becomes a space, which collapses punctuation
   differences — `Lip-Bu Tan` / `Lip Bu Tan`, `Dillard's` / `Dillards`, `3M` / `3-M`;
5. runs of whitespace collapse to one space.

Matching is then plain substring containment **on space-padded strings**, which makes it a
whole-word test: it stops `Ball` (the packaging company) from matching *"basketball"* or
*"Ballmer"*, and `Dana` from matching *"Danaher"*. It is deliberately *not* a per-token
subset test — the words have to be adjacent and in order, so `Michael Roman` does not match
a title that merely mentions a Michael and a Roman separately.

**A name that normalises to nothing is dropped** (an empty needle would match every title).

In [2]:
def normalize(text) -> str:
    """Fold a title or a name to the common form used for comparison."""
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    # decompose, then drop the combining marks -> "é" becomes "e"
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower().replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def contains_name(haystack_padded: str, needle_norm: str) -> bool:
    """Whole-word containment. `haystack_padded` is a normalised title wrapped in spaces."""
    return f" {needle_norm} " in haystack_padded


def split_alt_names(value) -> list[str]:
    """`"Carl Lindner III; Carl Lindner"` -> ["Carl Lindner III", "Carl Lindner"]."""
    if value is None or pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(ALT_SEPARATOR) if part.strip()]


def name_variants(*values) -> list[str]:
    """Normalise every spelling, drop blanks and duplicates, keep the order given."""
    variants, seen = [], set()
    for value in values:
        norm = normalize(value)
        if norm and norm not in seen:
            seen.add(norm)
            variants.append(norm)
    return variants

## 3. Build the name lookup from `ceos.csv`

One entry per **search** — i.e. per record stage 1 produced. A company with a co-CEO in a given
year contributes two entries for the same `(year, company)`, kept apart by the slot, exactly
the way `get_raw_videos.ipynb` keys its records.

Each entry carries the CEO spellings, the company spellings, and the canonical CEO name that
identifies this person in the statistics below.

In [3]:
def record_key(year, company, slot: str = "first") -> tuple:
    """Unique id for one search. Mirrors the keying used in get_raw_videos.ipynb."""
    return (int(year), str(company), slot)


ceos = pd.read_csv(CEOS_CSV)

# Build {search key -> {ceo variants, company variants, canonical ceo name}}
search_names = {}
for _, row in ceos.iterrows():
    company_variants = name_variants(row["company"], row["company_alt_names"])

    for slot, ceo_col, alt_col in (("first",  "CEO",        "CEO_alt_names"),
                                   ("second", "second_CEO", "second_CEO_alt_names")):
        ceo = row[ceo_col]
        if pd.isna(ceo) or not str(ceo).strip():
            continue          # no name -> stage 1 ran no search for this slot
        search_names[record_key(row["year"], row["company"], slot)] = {
            "ceo": name_variants(ceo, *split_alt_names(row[alt_col])),
            "company": company_variants,
            "canonical_ceo": str(ceo).strip(),
        }

n_second = sum(1 for k in search_names if k[2] == "second")
print(f"Rows in ceos.csv          : {len(ceos):,}")
print(f"Searches with a CEO name  : {len(search_names):,}"
      f"  ({len(search_names) - n_second:,} CEO + {n_second:,} second_CEO)")
print(f"Unique CEO names          : {len({v['canonical_ceo'] for v in search_names.values()}):,}")

Rows in ceos.csv          : 3,000
Searches with a CEO name  : 3,039  (3,000 CEO + 39 second_CEO)
Unique CEO names          : 920


## 4. Load the raw videos

The `.jsonl` checkpoint is append-only, so a pair that was re-collected (because the CEO name
changed in `ceos.csv`) appears more than once. **Later lines win** — the same rule
`load_progress()` uses in stage 1 — so the record kept here is always the one collected under
the current name.

In [4]:
def load_raw_records(path: Path) -> dict:
    """Read the append-only checkpoint into {search key -> record}. Later lines win."""
    records, n_lines = {}, 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            n_lines += 1
            rec = json.loads(line)
            records[record_key(rec["year"], rec["company"],
                               rec.get("ceo_slot") or "first")] = rec
    return records, n_lines


raw_records, n_lines = load_raw_records(INPUT_JSONL)
n_raw_videos = sum(len(rec.get("videos") or []) for rec in raw_records.values())

print(f"Lines in {INPUT_JSONL.name:<20}: {n_lines:,}")
print(f"Searches after dedup       : {len(raw_records):,}"
      f"  ({n_lines - len(raw_records):,} superseded re-collections dropped)")
print(f"Videos                     : {n_raw_videos:,}")

unknown = [k for k in raw_records if k not in search_names]
if unknown:
    print(f"\n⚠ {len(unknown):,} searches have no matching row in ceos.csv "
          f"and cannot be filtered: {unknown[:5]}")

Lines in all_videos.jsonl    : 3,674
Searches after dedup       : 3,039  (635 superseded re-collections dropped)
Videos                     : 104,641


## 5. Apply the filter

For every video the normalised title is tested against every spelling of the CEO and of the
company, and against the two CEO markers. The kept videos get a `match` block recording which
path matched and on which spelling — that is what makes a spot-check of the borderline cases
possible later without re-running anything.

The **CEO name alone** is sufficient: a title naming the person is about the person, whether or
not it also says "CEO". The **company alone is not** — it needs the marker beside it. This is
what keeps *"Nike CEO: We're putting the athlete back at the center"* (company + marker, no
name) while dropping *"Apple Is Falling Apart (On Purpose)"* (company, no marker, no name).

The cost is real and worth stating: a title that refers to the CEO only obliquely and never
says "CEO" — *"Jassy Says He Wants Amazon to Be Like a Startup"*, where the surname alone
appears but not as an `alt_name` — is now lost. That is the trade being made deliberately, in
exchange for the several thousand company-news videos the marker rule removes.

In [5]:
def match_title(title: str, names: dict) -> dict | None:
    """Which spellings occur in `title`? None if it fails both paths.

    Path A — any CEO spelling is present.
    Path B — any company spelling is present AND a CEO marker is present.
    """
    padded = f" {normalize(title)} "

    ceo_hits = [v for v in names["ceo"] if contains_name(padded, v)] if MATCH_ON_CEO else []
    company_hits = ([v for v in names["company"] if contains_name(padded, v)]
                    if MATCH_ON_COMPANY else [])
    marker_hits = [m for m in CEO_MARKERS if contains_name(padded, m)]

    if ceo_hits:
        path = "ceo"
    elif company_hits and marker_hits:
        path = "company+marker"
    else:
        return None                    # bare company, or nothing at all

    return {
        "on": path,
        "ceo_names": ceo_hits,
        "company_names": company_hits,
        "ceo_markers": marker_hits,
    }


filtered_records = {}
for key, rec in raw_records.items():
    names = search_names.get(key)
    if names is None:
        continue                       # not in ceos.csv — cannot be filtered, so excluded

    kept = []
    for video in rec.get("videos") or []:
        title = (video.get("snippet") or {}).get("title", "")
        match = match_title(title, names)
        if match:
            kept.append({**video, "match": match})

    filtered_records[key] = {**rec,
                             "canonical_ceo": names["canonical_ceo"],
                             "n_videos_kept": len(kept),
                             "videos": kept}

filtered_videos = [v for rec in filtered_records.values() for v in rec["videos"]]
print(f"Kept {len(filtered_videos):,} / {n_raw_videos:,} videos "
      f"({len(filtered_videos) / n_raw_videos:.1%})")

Kept 15,716 / 104,641 videos (15.0%)


## 6. Save

In [6]:
tmp = OUTPUT_JSONL.with_suffix(".jsonl.tmp")
with open(tmp, "w", encoding="utf-8") as f:
    for key in sorted(filtered_records,
                      key=lambda k: (-k[0], filtered_records[k].get("rank") or 9999, k[2] != "first")):
        f.write(json.dumps(filtered_records[key], ensure_ascii=False) + "\n")
tmp.replace(OUTPUT_JSONL)   # atomic — never leaves a half-written file behind

print(f"Wrote {len(filtered_records):,} records to {OUTPUT_JSONL} "
      f"({OUTPUT_JSONL.stat().st_size / 1e6:.1f} MB)")

Wrote 3,039 records to data/output/videos_metadata/filtered_videos.jsonl (58.9 MB)


## 7. Summary statistics

- **Unique CEOs** counts distinct canonical names, pooling `CEO` and `second_CEO` — one person
  who was CEO in four years is one CEO, and a co-CEO counts the same as any other.
- **Videos per search** divides by the searches that **kept at least one video**. Empty
  searches are excluded from the denominator, so this average is always ≥ 1 and describes the
  size of a typical *surviving* search rather than being diluted by the pairs that yielded
  nothing. The average over all searched pairs is printed underneath as context.
- **Average length** is over all kept videos. Durations come from `contentDetails.duration`
  (ISO-8601); the handful of videos missing one are excluded and reported.
- **Matched on** splits the survivors by which of the two paths kept them — the `ceo` path
  (a name in the title) or the `company+marker` path. The second number is the one to watch:
  it is the share of the set resting on *"[Company] CEO ..."* phrasing rather than on a name.

In [7]:
# ── Video counts ─────────────────────────────────────────────────────────────
n_before = n_raw_videos
n_after = len(filtered_videos)

# ── Unique CEOs with at least one surviving video (CEO and second_CEO pooled) ─
ceos_with_videos = {rec["canonical_ceo"] for rec in filtered_records.values() if rec["videos"]}

# ── Videos per search (one search = one CEO/year pair) ────────────────────────
n_searches = len(filtered_records)
n_searches_nonempty = sum(1 for rec in filtered_records.values() if rec["videos"])

# ── Duration ─────────────────────────────────────────────────────────────────
durations = []
n_no_duration = 0
for video in filtered_videos:
    iso = (video.get("contentDetails") or {}).get("duration")
    try:
        durations.append(isodate.parse_duration(iso).total_seconds())
    except Exception:
        n_no_duration += 1


def hms(seconds: float) -> str:
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h}h {m:02d}m {s:02d}s" if h else f"{m}m {s:02d}s"


print("=" * 66)
print("FILTER SUMMARY — title names the CEO, or the company + a CEO marker".center(66))
print("=" * 66)
print(f"Videos before filtering      : {n_before:>10,}")
print(f"Videos after filtering       : {n_after:>10,}   ({n_after / n_before:.1%} kept)")
print(f"Unique CEOs with 1+ video    : {len(ceos_with_videos):>10,}")
print(f"Avg videos per search        : {n_after / n_searches_nonempty:>13.2f}"
      f"   (over the {n_searches_nonempty:,} searches that kept 1+ video)")
print(f"Average video length         : {hms(mean(durations)):>10}"
      f"   ({mean(durations):,.0f} s over {len(durations):,} videos)")

# ── Context for the headline numbers ─────────────────────────────────────────
all_ceos = {v["canonical_ceo"] for v in search_names.values()}
by_reason = collections.Counter(v["match"]["on"] for v in filtered_videos)
also_company = sum(1 for v in filtered_videos
                   if v["match"]["on"] == "ceo" and v["match"]["company_names"])

print()
print("-" * 66)
print(f"CEOs searched                : {len(all_ceos):>10,}"
      f"   ({len(all_ceos) - len(ceos_with_videos):,} kept no videos)")
print(f"Searches keeping 1+ video    : {n_searches_nonempty:>10,} / {n_searches:,}"
      f"   ({n_searches - n_searches_nonempty:,} kept nothing)")
print(f"Avg over all searched pairs  : {n_after / n_searches:>13.2f}")
print(f"Median video length          : {hms(median(durations)):>10}")
if n_no_duration:
    print(f"Videos with no duration      : {n_no_duration:>10,}   (excluded from the average)")
print()
print("Kept by ...")
print(f"  {'CEO name in the title':<26}: {by_reason['ceo']:>10,}"
      f"   ({by_reason['ceo'] / n_after:.1%})")
print(f"  {'company + CEO marker':<26}: {by_reason['company+marker']:>10,}"
      f"   ({by_reason['company+marker'] / n_after:.1%})")
print(f"  {'(of the name matches, also':<26}  {also_company:>10,} name the company)")

FILTER SUMMARY — title names the CEO, or the company + a CEO marker
Videos before filtering      :    104,641
Videos after filtering       :     15,716   (15.0% kept)
Unique CEOs with 1+ video    :        682
Avg videos per search        :          8.69   (over the 1,809 searches that kept 1+ video)
Average video length         :    15m 29s   (929 s over 15,687 videos)

------------------------------------------------------------------
CEOs searched                :        920   (238 kept no videos)
Searches keeping 1+ video    :      1,809 / 3,039   (1,230 kept nothing)
Avg over all searched pairs  :          5.17
Median video length          :     5m 00s
Videos with no duration      :         29   (excluded from the average)

Kept by ...
  CEO name in the title     :     13,056   (83.1%)
  company + CEO marker      :      2,660   (16.9%)
  (of the name matches, also       3,862 name the company)


---

# Layer 2 — Categorise the videos into five CSR themes

Layer 1 kept every video whose **title** names the CEO, or names the company beside a CEO
marker. This second pass sorts those videos into five themes of corporate social
responsibility. A video can land in **one or more** categories, or in none — the ones matching
nothing are dropped.

| # | category | key | what it covers |
|---|---|---|---|
| 1 | Diversity, Equity & Inclusion | `DEI` | representation and treatment by race, gender, orientation, disability, age, background |
| 2 | Environmental sustainability | `environment` | climate, emissions, energy transition, pollution, waste, water, nature |
| 3 | Worker welfare & Corporate governance | `worker_governance` | pay, safety, unions, layoffs; board accountability, CEO succession, ethics, corruption, human rights |
| 4 | Customer responsibility | `customer_responsibility` | product safety, privacy, honest pricing and advertising, vulnerable users |
| 5 | Innovation and technology responsibility | `tech_responsibility` | AI and its ethics, automation and jobs, algorithmic bias, tech regulation |

### Title-only, and no target to hit

**The description is not searched.** An earlier version of this notebook searched title *and*
description, because title-only matching returned far fewer videos and the DEI count in
particular fell short of what an earlier strict pass had reached. Chasing that number was the
mistake: the description is where uploader boilerplate lives, so the extra volume it bought was
substantially off-topic, and a long list of terms had to be deleted one by one to hold the
boilerplate back.

There is now **no quota**. The categorisation is a statement about what a video's title is
about, and whatever number of videos satisfies that is the number the analysis gets. In
exchange, the lists can be *more* generous than before, because the failure mode the deletions
were defending against — boilerplate — is gone with the description.

**Output:** `data/output/videos_metadata/topic_filtered_videos.json`.

## 9. The keyword files

One JSON file per category in `data/keywords/`. Each has a `category` key (the short id), a
`label`, a `description`, `notes` recording the judgement calls, and `keywords` — a dict of
**named groups** of surface forms:

```json
{
  "category": "environment",
  "label": "Environmental sustainability",
  "keywords": {
    "climate":   ["climate", "climate change", "global warming", "..."],
    "emissions": ["carbon", "net zero", "decarbonization", "..."]
  }
}
```

Grouping earns its keep in the statistics: it records *why* a video was categorised, so a
category resting entirely on one broad group is visible rather than hidden.

Matching is the same mechanism as layer 1 — `normalize()` then space-padded containment — so
terms match as **standalone words**, case- and accent-insensitively, and multi-word entries
match as phrases. Adding a term is just editing the JSON; no code changes.

### Terms restored now that the description is out of scope

Seven terms had been deleted for one reason only: they were being triggered by text the
uploader pastes into every description. In a **title** that failure mode does not exist, so the
ones that are unambiguous there are back, and the qualifying phrases that had replaced them
stay alongside:

| restored | the boilerplate it used to hit | in titles |
|---|---|---|
| `privacy` | *"omnystudio.com/listener for privacy information"* | 16 hits, all genuinely about privacy — *"Sundar Pichai on Future of AI, Antitrust, and Privacy"* |
| `copyright` | *"© 2024 … fair use … Copyright Act 1976"* | 0 hits today; restored so a re-collection is not silently blind to it |
| `surveillance` | *"Bloomberg **Surveillance**"* — the TV show | 0 hits today; same reasoning |
| `disclosure` | *"affiliate disclosure: I may earn a commission"* | 0 hits today; same reasoning |

Three stay out, because in titles they are **still ambiguous** rather than boilerplate:
`ecosystem` (*"payment ecosystem"*, *"omnichannel ecosystem"* — 5 of 5 title hits off-topic),
`overtime` (*"CNBC's Closing Bell Overtime"*), and `subscribers` (a business metric, not a
customer-responsibility subject).

### Terms added

Measured against the layer-1 titles before being added — each is listed with its title hit
count and the reason it belongs:

| category | added | hits | why |
|---|---|---|---|
| `worker_governance` | a new **leadership transition** group: `steps down`, `step down`, `stepping down`, `resigns`, `resignation`, `retire`, `retires`, `retirement`, `ousts`, `fired`, `new ceo`, `next ceo`, `interim ceo`, `named ceo`, `successor`, `succeeds`, … | 260 | CEO succession is governance, and the group's own `succession` term was already there while every *other* way of saying it was missing |
| `worker_governance` | an **oversight** group: `testifies`, `testimony`, `senate hearing`, `senate committee`, `before congress`, `grilled`, … | 38 | a CEO answering to a legislature is the accountability the category is named for |
| `worker_governance` | `hiring`, `hire`, `hires`, `new hires` | 55 | the `employees` group had `headcount` and `attrition` but no word for hiring |
| `customer_responsibility` | `privacy` (restored, above) | 16 | |
| `environment` | `natural gas`, `lng`, `electric car`, `electric cars` | 20 | `fossil fuel` and `coal` were in, the two commonest fossil-fuel words were not |
| `tech_responsibility` | `chip`, `chips` | 75 | dropped once as "ordinary product talk"; in a CEO title they are the semiconductor and AI-infrastructure story, which is what the `infrastructure` group is for |

### Terms considered and rejected

Each was measured and left out, so the decision is on the record rather than an oversight:

| rejected | hits | why not |
|---|---|---|
| `leadership`, `management` | 278 / 52 | generic leadership-advice content — *"Decide What to Own, Delegate the Rest"*. Adding it would re-import the exact material this revision removes |
| `jobs`, `job` | 78 / 71 | **36 of the 78** are *Steve Jobs*. The AI-and-work sense is already carried by `job displacement`, `ai jobs`, `future of work` |
| `tariffs`, `inflation`, `price` | 36 / 60 / 31 | trade and macro policy, not a company's responsibility to its customers. Bare `price` also hits *"share price"* and the surname *Price* |
| `oil`, `gas` | 68 / 35 | the oil *business*, not its environmental impact — *"Oil Falls As Tariff Fears Hit Demand"*. The qualified `natural gas` and `lng` are in instead |
| `impact`, `power`, `china`, `trump` | 124 / 93 / 100 / 133 | no category of their own; each would attach a large slice of general business news to whichever theme it was filed under |

### The disambiguation decisions that still hold

These were made against the description corpus, and were **re-measured against titles** for
this revision — all four remain right:

| still out | title hits | what they match |
|---|---|---|
| `black` | 63 | *ADP CEO **Maria Black***, *Jeff Miller of **Black** Crystal Wolf Kids* — kept instead: `black employees`, `african american`, `black owned`, … |
| `environment` | 18 | *"a lower-priced **environment**"*, *"banking **environment**"* — kept instead: `environmental impact`, `environmental protection`, … |
| `water` | 7 | *"GM CEO Mary Barra in Hot **Water**?"* — kept instead: `water scarcity`, `water quality`, `clean water`, … |
| `talent` | 27 | *"Why Hard Work Beats **Talent**"* — kept instead: `talent retention`, `talent development` |

Two known-generous inclusions remain, both reported group-by-group below: bare
**`consumer`/`customer`** in category 4 also catches ordinary demand commentary (*"the consumer
is strong"*), and bare **`ai`** in category 5 catches all AI talk, not only its ethical
dimension.

In [8]:
KEYWORDS_DIR = Path("data/keywords")
TOPIC_OUTPUT_JSON = OUTPUT_DIR / "topic_filtered_videos.json"

# Layer 2 searches the TITLE ONLY — the description is uploader boilerplate as often as it is
# content, and matching on it was the source of the off-topic tail. See the note above.
TOPIC_SEARCH_FIELDS = ("title",)

# Display order for the five categories (file order is alphabetical, which is not the order
# the categories are numbered in).
CATEGORY_ORDER = ["DEI", "environment", "worker_governance",
                  "customer_responsibility", "tech_responsibility"]


def load_keyword_files(directory: Path) -> dict:
    """Read every category JSON in `directory` into {category: {label, groups, n_terms}}."""
    categories = {}
    for path in sorted(directory.glob("*.json")):
        spec = json.load(open(path, encoding="utf-8"))
        groups = {}
        for group, forms in spec["keywords"].items():
            # Longest first, so the term reported for a hit is the most specific one that fits.
            terms = sorted({normalize(t) for t in forms} - {""}, key=len, reverse=True)
            if terms:
                groups[group] = terms
        categories[spec["category"]] = {
            "label": spec["label"],
            "file": path.name,
            "groups": groups,
            "n_terms": sum(len(t) for t in groups.values()),
        }
    return categories


CATEGORIES = load_keyword_files(KEYWORDS_DIR)

missing = [c for c in CATEGORY_ORDER if c not in CATEGORIES]
extra = [c for c in CATEGORIES if c not in CATEGORY_ORDER]
if missing:
    raise FileNotFoundError(f"No keyword file defines: {missing} (looked in {KEYWORDS_DIR})")
CATEGORY_ORDER = CATEGORY_ORDER + sorted(extra)   # tolerate new files being added

print(f"Loaded {len(CATEGORIES)} categories from {KEYWORDS_DIR}/\n")
for cat in CATEGORY_ORDER:
    spec = CATEGORIES[cat]
    print(f"  {spec['label'][:44]:<46} {spec['n_terms']:>4} terms "
          f"in {len(spec['groups']):>2} groups   ({spec['file']})")
print(f"\n  {'total':<46} {sum(s['n_terms'] for s in CATEGORIES.values()):>4} terms")

Loaded 5 categories from data/keywords/

  Diversity, Equity & Inclusion                   245 terms in 26 groups   (dei.json)
  Environmental sustainability                    180 terms in 15 groups   (environment.json)
  Worker welfare & Corporate governance           281 terms in 18 groups   (worker_governance.json)
  Customer responsibility                         189 terms in 12 groups   (customer_responsibility.json)
  Innovation and technology responsibility        212 terms in 14 groups   (tech_responsibility.json)

  total                                          1107 terms


### Check the matching behaviour

The same standalone-word guarantee as layer 1: `"dei"` matches only as its own word, never
inside `design` or `redesign`. These assertions also pin down the disambiguation decisions
above — if someone re-adds bare `black`, bare `environment`, bare `water` or bare `leadership`
to a keyword file, the notebook says so instead of silently changing the numbers — and they
pin down the terms this revision *added*, so a later cleanup cannot quietly remove them.

In [9]:
def categorise(text: str) -> dict[str, dict[str, list[str]]]:
    """{category: {group: [terms found]}} for every category matching `text`."""
    padded = f" {normalize(text)} "
    result = {}
    for category, spec in CATEGORIES.items():
        groups = {}
        for group, terms in spec["groups"].items():
            found = [t for t in terms if f" {t} " in padded]
            if found:
                groups[group] = found
        if groups:
            result[category] = groups
    return result


def _cats(text) -> set:
    return set(categorise(text))


# Standalone-word matching — "dei" as its own word, not as a substring
assert "DEI" in _cats("Target CEO Resigns After DEI Backlash")
assert "DEI" in _cats("the (DEI) program"), "punctuation should not block a match"
assert "DEI" not in _cats("a redesign of the store"), "'design' must not match 'dei'"
assert "DEI" not in _cats("Deion Sanders interview")

# Each category fires on an obvious member of it
assert "environment" in _cats("Exxon Mobil CEO on EU climate regulations")
assert "worker_governance" in _cats("Amazon CEO on middle managers and return to office")
assert "customer_responsibility" in _cats("Ford recalls 100,000 vehicles over defect")
assert "tech_responsibility" in _cats("Nvidia CEO on responsible AI and job displacement")

# A video can belong to several categories at once
assert {"tech_responsibility", "worker_governance"} <= _cats("AI will replace workers, says CEO")

# The disambiguation decisions documented above stay made
assert "DEI" not in _cats("ADP CEO Maria Black on the labor market"), "bare 'black' is back"
assert "environment" not in _cats("prepared for a lower-priced environment"), "bare 'environment' is back"
assert "environment" not in _cats("GM CEO Mary Barra in Hot Water?"), "bare 'water' is back"
assert "DEI" in _cats("supporting Black employees"), "the qualified black phrases should still fire"

# The terms rejected in this revision stay rejected
assert not _cats("Walmart CEO on Leading with Purpose in Uncertain Times") & {"worker_governance"} \
    or "leadership" not in str(_cats("a decade of leadership")), "bare 'leadership' is back"
assert _cats("Buffett on Steve Jobs and Tim Cook") == set(), "bare 'jobs' is back — it matches Steve Jobs"
assert "customer_responsibility" not in _cats("Ford CEO on tariff turmoil"), "bare 'tariffs' is back"
assert "environment" not in _cats("Baker Hughes CEO Talks Oil Production"), "bare 'oil' is back"

# The terms added in this revision fire
assert "worker_governance" in _cats("Walmart CEO steps down"), "leadership-transition group missing"
assert "worker_governance" in _cats("Starbucks CEO Kevin Johnson to retire")
assert "worker_governance" in _cats("Meet Boeing's New CEO: Kelly Ortberg")
assert "worker_governance" in _cats("Boeing CEO testifies before the Senate committee")
assert "worker_governance" in _cats("Bank of America CEO Talks Strategy and Hiring")
assert "customer_responsibility" in _cats("Sundar Pichai on Future of AI, Antitrust, and Privacy")
assert "environment" in _cats("Natural gas and LNG play a key role in the energy tri-lemma")
assert "tech_responsibility" in _cats("Qualcomm CEO Says Intel's Chip Production Not Good Enough")

print("✓ standalone-word, multi-category, disambiguation and new-term checks pass")

✓ standalone-word, multi-category, disambiguation and new-term checks pass


## 10. Apply the categorisation

Each surviving video carries a `topics` block: the categories it matched, and per category the
groups and the exact terms. That is what the per-category statistics below are computed from,
and it means any single classification can be explained without re-running anything.

In [10]:
topic_records = {}
for key, rec in filtered_records.items():
    kept = []
    for video in rec["videos"]:
        snippet = video.get("snippet") or {}
        text = " \n ".join(snippet.get(f) or "" for f in TOPIC_SEARCH_FIELDS)
        matched = categorise(text)
        if not matched:
            continue
        kept.append({**video, "topics": {
            "categories": sorted(matched, key=CATEGORY_ORDER.index),
            "matches": {c: {g: terms for g, terms in groups.items()}
                        for c, groups in matched.items()},
        }})

    if kept:
        topic_records[key] = {**rec, "n_videos_topic": len(kept), "videos": kept}

topic_videos = [v for rec in topic_records.values() for v in rec["videos"]]
print(f"Kept {len(topic_videos):,} / {len(filtered_videos):,} videos "
      f"({len(topic_videos) / len(filtered_videos):.1%} of layer 1, "
      f"{len(topic_videos) / n_raw_videos:.1%} of the raw set)")

Kept 3,151 / 15,716 videos (20.0% of layer 1, 3.0% of the raw set)


## 11. Save

In [11]:
ordered = [topic_records[k] for k in sorted(
    topic_records, key=lambda k: (-k[0], topic_records[k].get("rank") or 9999, k[2] != "first"))]

tmp = TOPIC_OUTPUT_JSON.with_suffix(".json.tmp")
with open(tmp, "w", encoding="utf-8") as f:
    json.dump(ordered, f, ensure_ascii=False, indent=2)
tmp.replace(TOPIC_OUTPUT_JSON)   # atomic

print(f"Wrote {len(ordered):,} records / {len(topic_videos):,} videos to {TOPIC_OUTPUT_JSON} "
      f"({TOPIC_OUTPUT_JSON.stat().st_size / 1e6:.1f} MB)")

Wrote 906 records / 3,151 videos to data/output/videos_metadata/topic_filtered_videos.json (17.2 MB)


## 12. Summary statistics

**Overall** first, then **the same four measures per category**. In the per-category block a
video counts once for every category it matched, so the five category totals sum to more than
the overall total.

Because only the title is searched, every classification is visible in the one line of text a
reader of the video sees first — there is no video categorised on evidence buried in a
description. The per-category tables show which keyword group carried each classification,
which is the quickest way to spot a category resting on one broad term.

As in layer 1, *avg videos per search* divides by the searches that kept at least one video —
per category, that means the searches with at least one video **in that category** — so every
average is ≥ 1 and describes a typical surviving search.

In [12]:
def duration_seconds(videos) -> tuple[list, int]:
    """ISO-8601 durations in seconds, plus a count of the videos that had none."""
    seconds, missing = [], 0
    for video in videos:
        try:
            seconds.append(isodate.parse_duration(
                (video.get("contentDetails") or {})["duration"]).total_seconds())
        except Exception:
            missing += 1
    return seconds, missing


def block(videos, records, indent=""):
    """The four headline measures for a set of videos and the records holding them."""
    ceos = {rec["canonical_ceo"] for rec in records}
    n_nonempty = len(records)
    seconds, missing = duration_seconds(videos)
    print(f"{indent}Total videos                 : {len(videos):>10,}")
    print(f"{indent}Unique CEOs with 1+ video    : {len(ceos):>10,}")
    print(f"{indent}Avg videos per search        : {len(videos) / n_nonempty:>13.2f}"
          f"   (over the {n_nonempty:,} searches that kept 1+)")
    if seconds:
        print(f"{indent}Average video length         : {hms(mean(seconds)):>10}"
              f"   (median {hms(median(seconds))})")
    if missing:
        print(f"{indent}Videos with no duration      : {missing:>10,}   (excluded from the average)")


print("=" * 66)
print("TOPIC FILTER SUMMARY — overall".center(66))
print("=" * 66)
block(topic_videos, list(topic_records.values()))
print(f"{'Searches keeping 1+ video':<29}: {len(topic_records):>10,} / {n_searches:,}"
      f"   ({n_searches - len(topic_records):,} kept nothing)")
print(f"{'Avg over all searched pairs':<29}: {len(topic_videos) / n_searches:>13.2f}")

# ── How many categories a video lands in ─────────────────────────────────────
n_cats = collections.Counter(len(v["topics"]["categories"]) for v in topic_videos)
print()
print(f"{'Videos in exactly 1 category':<29}: {n_cats[1]:>10,}   ({n_cats[1] / len(topic_videos):.1%})")
for k in sorted(c for c in n_cats if c > 1):
    print(f"{'Videos in ' + str(k) + ' categories':<29}: {n_cats[k]:>10,}   ({n_cats[k] / len(topic_videos):.1%})")

# ── Funnel ───────────────────────────────────────────────────────────────────
print()
print(" Funnel ".center(66, "-"))
print(f"{'raw (stage 1)':<32}{n_raw_videos:>10,}")
print(f"{'+ title names CEO/company':<32}{n_after:>10,}   ({n_after / n_raw_videos:>6.1%} of raw)")
print(f"{'+ matches a CSR category':<32}{len(topic_videos):>10,}   "
      f"({len(topic_videos) / n_raw_videos:>6.1%} of raw)")

# ── Per category ─────────────────────────────────────────────────────────────
for i, category in enumerate(CATEGORY_ORDER, start=1):
    cat_videos = [v for v in topic_videos if category in v["topics"]["categories"]]
    cat_records = [rec for rec in topic_records.values()
                   if any(category in v["topics"]["categories"] for v in rec["videos"])]
    print()
    print("=" * 66)
    print(f"{i}. {CATEGORIES[category]['label'].upper()}  [{category}]")
    print("=" * 66)
    if not cat_videos:
        print("  no videos matched this category")
        continue

    block(cat_videos, cat_records, indent="  ")
    print(f"  {'Share of categorised videos':<29}: {len(cat_videos) / len(topic_videos):>13.1%}")

    # Overlap with the other four
    overlap = collections.Counter(
        other for v in cat_videos for other in v["topics"]["categories"] if other != category)
    only = sum(1 for v in cat_videos if len(v["topics"]["categories"]) == 1)
    print(f"  {'Only this category':<29}: {only:>10,}   ({only / len(cat_videos):.1%})")
    if overlap:
        shared = ", ".join(f"{c} {n:,}" for c, n in overlap.most_common(4))
        print(f"  {'Also matched':<29}: {shared}")

    # Which keyword groups carry the category
    group_hits = collections.Counter(
        g for v in cat_videos for g in v["topics"]["matches"][category])
    sole_group = collections.Counter(
        next(iter(v["topics"]["matches"][category]))
        for v in cat_videos if len(v["topics"]["matches"][category]) == 1)
    print(f"\n  {'keyword group':<26}{'videos':>9}{'only reason':>13}")
    print("  " + "-" * 48)
    for group, n in group_hits.most_common(8):
        print(f"  {group:<26}{n:>9,}{sole_group[group]:>13,}")
    silent = [g for g in CATEGORIES[category]["groups"] if not group_hits[g]]
    if silent:
        print(f"  ({len(silent)} groups matched nothing: {', '.join(sorted(silent))})")

                  TOPIC FILTER SUMMARY — overall                  
Total videos                 :      3,151
Unique CEOs with 1+ video    :        407
Avg videos per search        :          3.48   (over the 906 searches that kept 1+)
Average video length         :    12m 10s   (median 4m 24s)
Searches keeping 1+ video    :        906 / 3,039   (2,133 kept nothing)
Avg over all searched pairs  :          1.04

Videos in exactly 1 category :      2,953   (93.7%)
Videos in 2 categories       :        191   (6.1%)
Videos in 3 categories       :          7   (0.2%)

----------------------------- Funnel -----------------------------
raw (stage 1)                      104,641
+ title names CEO/company           15,716   ( 15.0% of raw)
+ matches a CSR category             3,151   (  3.0% of raw)

1. DIVERSITY, EQUITY & INCLUSION  [DEI]
  Total videos                 :        308
  Unique CEOs with 1+ video    :        125
  Avg videos per search        :          1.70   (over the 181 searche